## Hybrid chunking

##  | Modality          | Chunking Type         |
    | ----------------- | --------------------- |
    | Docling JSON text | Recursive + semantic  |
    | Tables (CSV)      | One table = one chunk |
    | Images            | One image = one chunk |
    | Charts/graphs     | Image + OCR + caption |
    | OCR pages         | Page-wise chunks      |
    | Metadata          | One per document      |


In [35]:
from pathlib import Path
import os
import re
import json
from typing import List, Dict, Any
import numpy as np
import pandas as pd
from tqdm import tqdm
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings

In [2]:
## JSON Chunking (Recursive + Semantic)
JSON_DIR = Path("../data/extracted/json")
CHUNKS_DIR = Path("../data/chunks/text")
CHUNKS_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
def load_docling_json(jsonpath : Path) -> Dict:
    with open(jsonpath, "r", encoding="utf-8") as f:
        return json.load(f)

In [4]:
def extract_paragraphs(docling_json: Dict) -> List[Dict]:
    #cleans docling’s complex structure
    texts_data = docling_json.get("texts",[])#List[Dict]
    paragraphs=[]
    for text_element in texts_data:
        text = text_element.get("text", "").strip()
        if len(text) > 20:
            provenence = text_element.get("prov")
            page_no = None

            if provenence and isinstance(provenence, list) and len(provenence) > 0:
                page_no = provenence[0].get("page_no")
            
            content_layer = text_element.get("content_layer")
            if content_layer in ["body", "unspecified"] and page_no is not None:
                paragraphs.append(
                    {
                        "text": text,
                        "page": page_no
                    }
                )
    
    return paragraphs



In [5]:
# docling_json = {
#     "texts": [
#         # Item 1: A Page Header (Furniture)
#         {
#             "text": "Data in Brief 62 (2025) 111938", 
#             "content_layer": "furniture",
#             "prov": [{"page_no": 1}]
#         },
#         # Item 2: A Valid Body Paragraph
#         {
#             "text": "The dataset provides clearly annotated manufacturing specifications.", 
#             "content_layer": "body",
#             "prov": [{"page_no": 2}]
#         }
#     ]
# }
# text_data
# [
#   {"text": "Data in Brief...", "content_layer": "furniture", ...},
#   {"text": "The dataset provides...", "content_layer": "body", ...}
# ]

In [6]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 700,
    chunk_overlap = 100,
    separators = ["\n\n", "\n", ". ", " ", ""]
)

In [7]:
embedding_model = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
)

/var/folders/bc/p_nnrzps7wdgbtdptyv7hg_h0000gn/T/ipykernel_12788/118603400.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(


In [8]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [9]:
def chunk_docling_json(json_path : Path) ->List[Dict]:
    raw = load_docling_json(json_path)
    paragraphs = extract_paragraphs(raw)

    chunks = []
    for para in paragraphs:
        splits = text_splitter.split_text(para["text"])
        for s in splits:
            chunks.append(
                {
                    "text": s,
                    "page": para["page"]
                }
            )
    
    #semantic merge
    texts = []
    def normalize_text(t: str) -> str:
        return " ".join(t.lower().split())
    for c in chunks:
        texts.append(normalize_text(c["text"]))
    #Convert text → embeddings
    embeddings = embedding_model.embed_documents(texts)

    final_chunk = []
    buffer = chunks[0]
    buffer_emb = embeddings[0]

    # merge_log = []

    for i in range(1, len(chunks)):
        sim = cosine_similarity(buffer_emb, embeddings[i])

        if sim > 0.85:
            #same topic merge

            # merge_log.append({
            #     "merged_at_index": i,
            #     "page": chunks[i]["page"],
            #     "similarity": round(sim, 3),
            #     "prev_text_preview": buffer["text"][:120],
            #     "merged_text_preview": chunks[i]["text"][:120]
            # })
            buffer["text"] += " " + chunks[i]["text"]
            buffer_emb = embedding_model.embed_documents([buffer["text"]])[0]
        else:
            final_chunk.append(buffer)
            buffer = chunks[i]
            buffer_emb = embeddings[i]

    final_chunk.append(buffer)

    # print("Total merges:", len(merge_log))
    # for m in merge_log:
    #     print(
    #         f"Merge at chunk {m['merged_at_index']} | "
    #         f"page {m['page']} | sim={m['similarity']}"
    #     )

    return final_chunk


In [10]:
# [
#   {"text": "Pump pressure exceeds safe limit...", "page": 2},
#   {"text": "Valve failure occurs during overload...", "page": 3}
# ]

# paragraphs = extract_paragraphs(raw)
# paragraphs:

# [
#   {"text": "Pump pressure exceeds...", "page": 2},
#   {"text": "This occurs during overload...", "page": 2}
# ]

# splits = text_splitter.split_text(para["text"])
# What happens
# Breaks long paragraphs into ~700-char chunks

# Uses:

# paragraph

# sentence

# word boundaries

# 📦 splits:

# [
#   "Pump pressure exceeds safe limit...",
#   "This occurs during overload..."
# ]
# chunks:

# [
#   {"text": "Pump pressure exceeds...", "page": 2},
#   {"text": "This occurs during overload...", "page": 2}
# ]
# texts:

# [
#   "Pump pressure exceeds...",
#   "This occurs during overload..."
# ]
# embeddings:

# [
#   [0.12, -0.33, ...],   # chunk 0
#   [0.14, -0.30, ...]    # chunk 1
# ]
# if sim > 0.85:
#     buffer["text"] += " " + chunks[i]["text"]
# 📦 Result:


# "Pump pressure exceeds... This occurs during overload..."


# final
# [
#   {"text": "Merged chunk about pump pressure...", "page": 2},
#   {"text": "New topic about valve failure...", "page": 3}
# ]


In [11]:
json_path = JSON_DIR/"sample_docling.json"
all_chunks = []
doc_chunks = chunk_docling_json(json_path)

for i, ch in enumerate(doc_chunks):
    all_chunks.append({
        "chunk_id": f"{json_path.stem}_{i}",
        "source": json_path.name.replace("_docling.json", ".pdf"),
        "page": ch["page"],
        "type": "text",
        "content": ch["text"]
    })


In [12]:
out_path = CHUNKS_DIR / "docling_text_chunks.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(all_chunks, f, indent=2)

In [13]:
#chunking tables

def load_table(csv_path: Path) -> pd.DataFrame:
    return pd.read_csv(csv_path)


In [25]:
def table_to_text(df: pd.DataFrame, max_rows: int) -> List[str]:
    chunks = []
    headers = " | ".join(df.columns)

    for start in range(0, len(df), max_rows):
        window = df.iloc[start:start + max_rows]

        rows_text = []

        for _, row in window.iterrows():
            rows_text.append(" | ".join(map(str, row.values)))
        
        chunk_text = (
            f"Table Columns:\n{headers}\n\n"
            f"Rows:\n" +"\n".join(rows_text)
        )

        chunks.append(chunk_text)
    return chunks

In [24]:
def chunk_table_csv(
    csv_path: Path,
    source_pdf: str,
    rows_per_chunk: int = 10
) -> List[Dict]:

    df = load_table(csv_path)
    table_chunks = table_to_text(df, rows_per_chunk)

    chunks = []
    for i, text in enumerate(table_chunks):
        chunks.append({
            "chunk_id": f"{csv_path.stem}_rows_{i}",
            "type": "table",
            "source": source_pdf,
            "content": text
        })

    return chunks


In [23]:
def chunk_all_tables(csv_dir: Path, source_pdf: str) -> List[Dict]:
    all_chunks = []

    for csv_file in csv_dir.glob("*.csv"):
        table_chunks = chunk_table_csv(
            csv_file,
            source_pdf,
            rows_per_chunk=10
        )
        all_chunks.extend(table_chunks)

    return all_chunks


In [20]:
table_chunks = chunk_all_tables(
    csv_dir=Path("../data/extracted/tables"),
    source_pdf="sample.pdf"
)

In [31]:
# for i, chunk in enumerate(table_chunks):
#     print(f"\n===== TABLE CHUNK {i} =====")
#     print(chunk["content"])


In [33]:
def chunk_images(image_dir: Path, source_pdf: str) -> List[Dict]:
    #One image = one chunk
    image_chunks = []
    supported_ext = {".png", ".jpg", ".jpeg", ".webp"}
    for idx, img_path in enumerate(sorted(image_dir.iterdir())):
        if img_path.suffix.lower() not in supported_ext:
            continue
        chunk = {
            "chunk_id": f"{Path(source_pdf).stem}_img_{idx}",
            "type": "image",
            "modality": "image",
            "image_path": str(img_path),
            "source": source_pdf,
            "page": None,          
            "caption": None 
        }
        image_chunks.append(chunk)

    print(f"Image chunks created: {len(image_chunks)}")
    return image_chunks

In [ ]:
IMAGE_DIR = Path("../data/extracted/images")

image_chunks = chunk_images(
    image_dir=IMAGE_DIR,
    source_pdf="sample.pdf"
)
image_chunks[0]


Image chunks created: 51


{'chunk_id': 'sample_img_1',
 'type': 'image',
 'modality': 'image',
 'image_path': '../data/extracted/images/sample_p10_embed_0.jpeg',
 'source': 'sample.pdf',
 'page': None,
 'caption': None}

In [36]:
def clean_ocr_text(text: str) -> str:
    text = text.replace("\x0c", " ") 
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]{2,}", " ", text)
    return text.strip()



In [37]:
def chunk_ocr_text(
        ocr_dir: Path,
        source_pdf: str,
        chunk_size: int = 800,
        chunk_overlap: int = 100
    ) -> List[Dict]:

    ocr_chunks = []
    splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap
    )

    for ocr_file in sorted(ocr_dir.glob("*.txt")):
        raw_text = ocr_file.read_text(encoding="utf-8", errors="ignore")

        cleaned = clean_ocr_text(raw_text)
        if not cleaned:
            continue

        splits = splitter.split_text(cleaned)
        for idx, chunk in enumerate(splits):
            ocr_chunks.append({
                "chunk_id": f"{Path(source_pdf).stem}_ocr_{ocr_file.stem}_{idx}",
                "type": "text",
                "modality": "ocr",
                "content": chunk,
                "source": source_pdf,
                "page": None   # fill if OCR page mapping exists
            })

    print(f"OCR chunks created: {len(ocr_chunks)}")
    return ocr_chunks

In [40]:
OCR_DIR = Path("../data/extracted/ocr")

ocr_chunks = chunk_ocr_text(
    ocr_dir=OCR_DIR,
    source_pdf="sample.pdf"
)

# Inspect one chunk
ocr_chunks[240]


OCR chunks created: 242


{'chunk_id': 'sample_ocr_ocr9_1',
 'type': 'text',
 'modality': 'ocr',
 'content': 'cedure was implemented. Annotator 1 reviewed 100 examples from Annotator 2, Annotator 2\nreviewed 100 from Annotator 3, and Annotator 3 reviewed 100 from Annotator 1. This process\nsupported consistency checking across the team.\n\nThis review process helped identify disagreements and guided harmonization of annotation\npractices. Using this shared subset, Cohen’s Kappa was computed to measure inter-annotator\nagreement. The final average kappa score was 0.856, which indicates strong agreement.\n\nThe distribution of token-level annotations within the reviewed sample is provided in\nTable 12, while Table 13 presents the computed inter-annotator agreement scores.',
 'source': 'sample.pdf',
 'page': None}